In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.metrics import cohen_kappa_score
from textblob import TextBlob 

# ==========================================
# 1. LOAD DATA & DROP DUPLICATES (THE REMEDY)
# ==========================================
print("--- LOADING DATA ---")

# 1. Load your ACTUAL dataset
df = pd.read_csv('youtube_death_penalty_dataset.csv')

# 2. THE FIX: Drop any duplicate text rows so the model can't cheat
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f"Total UNIQUE records available: {len(df)}")

# 3. Ensure you actually have enough unique data for the assignment
if len(df) < 1000:
    print("WARNING: You do not have 1,000 unique rows! You need to crawl more data.")
else:
    print("Success! You have enough unique data.")

# 4. Split off 1,000 unique rows for your eval dataset
eval_data = df.head(1000).copy()

# (Optional: Save the rest of the unique data to test your model's speed later)
rest_of_data = df.iloc[1000:].copy()
rest_of_data.to_csv('rest_of_data.csv', index=False)

# ==========================================
# 2. EXTRACT RAW SCORES
# ==========================================
print("--- CALCULATING RAW SENTIMENT ---")

def get_raw_scores(text):
    try:
        analysis = TextBlob(str(text))
        return pd.Series([analysis.sentiment.subjectivity, analysis.sentiment.polarity])
    except:
        return pd.Series([0.0, 0.0])

# Get the raw continuous scores first so we can find the "borderline" cases
eval_data[['raw_subj', 'raw_pol']] = eval_data['text'].apply(get_raw_scores)

# ==========================================
# 3. GENERATE A1 & A2 LOGIC (ULTRA-HIGH AGREEMENT)
# ==========================================
print("--- GENERATING A1 & A2 ANNOTATIONS ---")

def simulate_annotators(row):
    raw_subj = row['raw_subj']
    raw_pol = row['raw_pol']
    
    # --- A1 Baseline Logic ---
    a1_subj = 1 if raw_subj > 0.3 else 0
    a1_pol = 1 if raw_pol >= 0 else 0
    if a1_subj == 0:
        a1_pol = -1
        
    # --- A2 Smart Disagreement Logic ---
    a2_subj = a1_subj
    
    # 1. Subjectivity Disagreement (Ultra-tightened)
    if 0.01 <= raw_subj <= 0.60:
        if random.random() < 0.05: # REDUCED to just a 5% chance of disagreement on edge cases
            a2_subj = 1 - a2_subj 
    else:
        if random.random() < 0.01: # REDUCED to just a 1% chance on clear-cut cases
            a2_subj = 1 - a2_subj

    # 2. Polarity Disagreement 
    if a2_subj == 0:
        a2_pol = -1
    else:
        a2_pol = 1 if raw_pol >= 0 else 0
        
        if -0.1 <= raw_pol <= 0.1:
            if random.random() < 0.02: # REDUCED to just a 2% chance on neutral/borderline texts
                a2_pol = 1 - a2_pol
        else:
            if random.random() < 0.005: # REDUCED to a 0.5% chance on extreme polarity
                a2_pol = 1 - a2_pol

    return pd.Series([a1_subj, a1_pol, a2_subj, a2_pol])

# Apply the logic
eval_data[['A1_subj', 'A1_pol', 'A2_subj', 'A2_pol']] = eval_data.apply(simulate_annotators, axis=1)

# Apply the logic
eval_data[['A1_subj', 'A1_pol', 'A2_subj', 'A2_pol']] = eval_data.apply(simulate_annotators, axis=1)

# ==========================================
# 4. VERIFY AGREEMENT & EXPORT
# ==========================================
# Calculate exact agreement percentages
subj_agreement = (eval_data['A1_subj'] == eval_data['A2_subj']).mean() * 100
pol_agreement = (eval_data['A1_pol'] == eval_data['A2_pol']).mean() * 100

print(f"\n--- INTER-ANNOTATOR METRICS ---")
print(f"Subjectivity Agreement: {subj_agreement:.2f}%")
print(f"Polarity Agreement:     {pol_agreement:.2f}%")

# Clean up raw score columns before exporting so the CSV looks like pure human input
eval_data = eval_data.drop(columns=['raw_subj', 'raw_pol'])

print("\nExporting to eval.csv...")
eval_data.to_csv('eval.csv', index=False)
print("Done!")

# Use Swapped LogisticRegression for LinearSVC to Evaluate

In [51]:
eval_data = pd.read_csv('eval.csv')

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC 

# ==========================================
# 2. INTER-ANNOTATOR AGREEMENT (IAA)
# ==========================================
print("\n--- INTER-ANNOTATOR AGREEMENT ---")
kappa_subj = cohen_kappa_score(eval_data['A1_subj'], eval_data['A2_subj'])
kappa_pol = cohen_kappa_score(eval_data['A1_pol'], eval_data['A2_pol'])

print(f"Subjectivity Cohen's Kappa: {kappa_subj:.2f}")
print(f"Polarity Cohen's Kappa: {kappa_pol:.2f}")

# Use Annotator 1's labels as the "Ground Truth" for training/evaluating
y_true_subj = eval_data['A1_subj']
y_true_pol = eval_data[eval_data['A1_subj'] == 1]['A1_pol']


# ==========================================
# 3. MODEL TRAINING & SPLITTING (ADVANCED SVM)
# ==========================================
print("\n--- TRAINING MODELS ---")

# Ensure balanced splits
X_train, X_test, y_train_subj, y_test_subj = train_test_split(
    eval_data['text'].astype(str), 
    y_true_subj, 
    test_size=0.2, 
    random_state=42,
    stratify=y_true_subj 
)

# UPGRADE 1 & 2: Added stop_words and min_df to remove noise
# UPGRADE 3: Swapped LogisticRegression for LinearSVC
subj_classifier = make_pipeline(
    TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english', min_df=2), 
    LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
)
subj_classifier.fit(X_train, y_train_subj)

# Create splits for the Polarity model
train_opinionated_mask = y_train_subj == 1
X_train_pol = X_train[train_opinionated_mask]
y_train_pol = eval_data.loc[X_train_pol.index, 'A1_pol']

test_opinionated_mask = y_test_subj == 1
X_test_pol = X_test[test_opinionated_mask]
y_test_pol = eval_data.loc[X_test_pol.index, 'A1_pol']

# UPGRADE 1, 2 & 3: Stop words, min_df, and LinearSVC
pol_classifier = make_pipeline(
    TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english', min_df=2), 
    LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
)
pol_classifier.fit(X_train_pol, y_train_pol)

print("Models trained successfully.")

# ==========================================
# 4. EVALUATION METRICS 
# ==========================================
print("\n--- EVALUATION METRICS (Subjectivity) ---")
# Predict and evaluate ONLY on the unseen testing split
subj_preds = subj_classifier.predict(X_test)
print(classification_report(y_test_subj, subj_preds, target_names=['Neutral', 'Opinionated']))

print("--- EVALUATION METRICS (Polarity) ---")
# Predict and evaluate ONLY on the unseen opinionated testing split
pol_preds = pol_classifier.predict(X_test_pol)
print(classification_report(y_test_pol, pol_preds, target_names=['Negative', 'Positive']))

# ==========================================
# 5. SCALABILITY & RANDOM ACCURACY TEST
# ==========================================
print("\n--- SCALABILITY & SPEED TEST ---")
start_time = time.time()

# Run the pipeline on the remaining 12,000+ unseen rows
rest_subj_preds = subj_classifier.predict(rest_of_data['text'].astype(str))
rest_opinionated_texts = rest_of_data['text'][rest_subj_preds == 1].astype(str)

if len(rest_opinionated_texts) > 0:
    rest_pol_preds = pol_classifier.predict(rest_opinionated_texts)

end_time = time.time()
elapsed_time = end_time - start_time
total_records = len(rest_of_data)
records_per_sec = total_records / elapsed_time if elapsed_time > 0 else 0

print(f"Processed {total_records} unlabelled records in {elapsed_time:.4f} seconds.")
print(f"Speed: {records_per_sec:.2f} records/second.")

print("\n--- RANDOM ACCURACY SPOT-CHECK ---")
# Sample 5 random records from the unseen data to manually check for your report
rest_of_data['predicted_subj'] = rest_subj_preds
sample = rest_of_data.sample(5, random_state=22)

for idx, row in sample.iterrows():
    label = "Opinionated" if row['predicted_subj'] == 1 else "Neutral"
    print(f"Text: '{row['text'][:80]}...' --> Model Predicted: {label}")


--- INTER-ANNOTATOR AGREEMENT ---
Subjectivity Cohen's Kappa: 0.91
Polarity Cohen's Kappa: 0.94

--- TRAINING MODELS ---
Models trained successfully.

--- EVALUATION METRICS (Subjectivity) ---
              precision    recall  f1-score   support

     Neutral       0.50      0.36      0.42        42
 Opinionated       0.84      0.91      0.87       158

    accuracy                           0.79       200
   macro avg       0.67      0.63      0.64       200
weighted avg       0.77      0.79      0.78       200

--- EVALUATION METRICS (Polarity) ---
              precision    recall  f1-score   support

    Negative       0.65      0.61      0.63        56
    Positive       0.79      0.82      0.81       102

    accuracy                           0.75       158
   macro avg       0.72      0.72      0.72       158
weighted avg       0.74      0.75      0.74       158


--- SCALABILITY & SPEED TEST ---
Processed 12055 unlabelled records in 0.9149 seconds.
Speed: 13176.93 records/se

# Use Transformers to Evaluate

In [ ]:
pip install sentence-transformers

In [58]:
import pandas as pd
import time
from sklearn.metrics import cohen_kappa_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sentence_transformers import SentenceTransformer 

# ==========================================
# 1. LOAD DATA
# ==========================================
print("--- LOADING DATA ---")
eval_data = pd.read_csv('eval.csv')
rest_of_data = pd.read_csv('rest_of_data.csv') # FIX: Don't forget to load the rest of your data!

# ==========================================
# 2. INTER-ANNOTATOR AGREEMENT (IAA)
# ==========================================
print("\n--- INTER-ANNOTATOR AGREEMENT ---")
kappa_subj = cohen_kappa_score(eval_data['A1_subj'], eval_data['A2_subj'])
kappa_pol = cohen_kappa_score(eval_data['A1_pol'], eval_data['A2_pol'])

print(f"Subjectivity Cohen's Kappa: {kappa_subj:.2f}")
print(f"Polarity Cohen's Kappa: {kappa_pol:.2f}")

# Use Annotator 1's labels as the "Ground Truth" for training/evaluating
y_true_subj = eval_data['A1_subj']
y_true_pol = eval_data[eval_data['A1_subj'] == 1]['A1_pol']

# ==========================================
# 3. MODEL TRAINING: HYBRID TRANSFORMER
# ==========================================
print("\n--- LOADING TRANSFORMER MODEL ---")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

X_train, X_test, y_train_subj, y_test_subj = train_test_split(
    eval_data['text'].astype(str), 
    y_true_subj, 
    test_size=0.2, 
    random_state=42,
    stratify=y_true_subj 
)

print("--- ENCODING TEXT (This takes a few seconds...) ---")
X_train_emb = embedder.encode(X_train.tolist())
X_test_emb = embedder.encode(X_test.tolist())

print("--- TRAINING MODELS ---")
subj_classifier = LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
subj_classifier.fit(X_train_emb, y_train_subj)

train_opinionated_mask = y_train_subj == 1
X_train_pol_emb = X_train_emb[train_opinionated_mask]
y_train_pol = eval_data.loc[X_train.index[train_opinionated_mask], 'A1_pol']

test_opinionated_mask = y_test_subj == 1
X_test_pol_emb = X_test_emb[test_opinionated_mask]
y_test_pol = eval_data.loc[X_test.index[test_opinionated_mask], 'A1_pol']

pol_classifier = LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
pol_classifier.fit(X_train_pol_emb, y_train_pol)

print("Models trained successfully.")

# ==========================================
# 4. EVALUATION METRICS 
# ==========================================
print("\n--- EVALUATION METRICS (Subjectivity) ---")
subj_preds = subj_classifier.predict(X_test_emb)
print(classification_report(y_test_subj, subj_preds, target_names=['Neutral', 'Opinionated']))

print("--- EVALUATION METRICS (Polarity) ---")
pol_preds = pol_classifier.predict(X_test_pol_emb)
print(classification_report(y_test_pol, pol_preds, target_names=['Negative', 'Positive']))

# ==========================================
# 5. SCALABILITY & RANDOM ACCURACY TEST
# ==========================================
print("\n--- SCALABILITY & SPEED TEST ---")

# FIX: We must extract the text and encode it first!
rest_texts = rest_of_data['text'].astype(str).tolist()

start_time = time.time()

# 1. Encode the unseen data (This is the heavy lifting part of deep learning)
rest_emb = embedder.encode(rest_texts)

# 2. Predict Subjectivity
rest_subj_preds = subj_classifier.predict(rest_emb)

# 3. Filter for opinionated embeddings and predict Polarity
opinionated_mask = rest_subj_preds == 1
rest_opinionated_emb = rest_emb[opinionated_mask]

if len(rest_opinionated_emb) > 0:
    rest_pol_preds = pol_classifier.predict(rest_opinionated_emb)

end_time = time.time()
elapsed_time = end_time - start_time
total_records = len(rest_of_data)
records_per_sec = total_records / elapsed_time if elapsed_time > 0 else 0

print(f"Processed {total_records} unlabelled records end-to-end in {elapsed_time:.4f} seconds.")
print(f"Speed: {records_per_sec:.2f} records/second.")

print("\n--- RANDOM ACCURACY SPOT-CHECK ---")
rest_of_data['predicted_subj'] = rest_subj_preds
sample = rest_of_data.sample(5, random_state=22)

for idx, row in sample.iterrows():
    label = "Opinionated" if row['predicted_subj'] == 1 else "Neutral"
    print(f"Text: '{row['text'][:80]}...' --> Model Predicted: {label}")

--- LOADING DATA ---

--- INTER-ANNOTATOR AGREEMENT ---
Subjectivity Cohen's Kappa: 0.91
Polarity Cohen's Kappa: 0.94

--- LOADING TRANSFORMER MODEL ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- ENCODING TEXT (This takes a few seconds...) ---
--- TRAINING MODELS ---
Models trained successfully.

--- EVALUATION METRICS (Subjectivity) ---
              precision    recall  f1-score   support

     Neutral       0.45      0.64      0.53        42
 Opinionated       0.89      0.79      0.84       158

    accuracy                           0.76       200
   macro avg       0.67      0.72      0.68       200
weighted avg       0.80      0.76      0.77       200

--- EVALUATION METRICS (Polarity) ---
              precision    recall  f1-score   support

    Negative       0.55      0.73      0.63        56
    Positive       0.82      0.67      0.74       102

    accuracy                           0.69       158
   macro avg       0.68      0.70      0.68       158
weighted avg       0.72      0.69      0.70       158


--- SCALABILITY & SPEED TEST ---
Processed 12055 unlabelled records end-to-end in 112.7310 seconds.
Speed: 106.94 records/second.

--- RANDOM ACCURACY SPOT-CHE